In [1]:
# Loome SparkSession-i koos Kafka konnektori ja Delta Lake'i toega.
# spark.jars.packages laeb vajalikud JAR-id Maven Central-ist Sparki käivitumisel.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("metalfab")
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.11")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}")

Spark 4.1.1


In [2]:
def write_to_postgres(batch_df, batch_id):
    db_url = "jdbc:postgresql://db:5432/praktikum"
    db_properties = {
       "user": "praktikum",
       "password": "praktikum",
       "driver": "org.postgresql.Driver"
        }
    # Kirjutame selle konkreetse partii andmebaasi
    batch_df.write.jdbc(
       url=db_url,
       table="bronze.raw_factory_data",
       mode="append",
       properties=db_properties
       )
    print(f"Batch {batch_id} kirjutatud andmebaasi.")

processed_df = spark.readStream \
    .schema("dept STRING, machine STRING, tag STRING, timestamp TIMESTAMP, topic STRING, value DOUBLE") \
    .json("../data/lake/*/*/*/*/*/*")

# 4. Käivita striim
query = processed_df.writeStream \
.foreachBatch(write_to_postgres) \
.option("checkpointLocation", "./checkpoints/postgres_stream") \
.start()


In [11]:
query.status

{'message': 'Initializing StreamExecution',
 'isDataAvailable': False,
 'isTriggerActive': False}

Batch 1 kirjutatud andmebaasi.
Batch 2 kirjutatud andmebaasi.
Batch 3 kirjutatud andmebaasi.
Batch 4 kirjutatud andmebaasi.
Batch 5 kirjutatud andmebaasi.
Batch 6 kirjutatud andmebaasi.
Batch 7 kirjutatud andmebaasi.
Batch 8 kirjutatud andmebaasi.
Batch 9 kirjutatud andmebaasi.
Batch 10 kirjutatud andmebaasi.
Batch 11 kirjutatud andmebaasi.
Batch 12 kirjutatud andmebaasi.
Batch 13 kirjutatud andmebaasi.
Batch 14 kirjutatud andmebaasi.
Batch 15 kirjutatud andmebaasi.
Batch 16 kirjutatud andmebaasi.
Batch 17 kirjutatud andmebaasi.
Batch 18 kirjutatud andmebaasi.
Batch 19 kirjutatud andmebaasi.
Batch 20 kirjutatud andmebaasi.
Batch 21 kirjutatud andmebaasi.
Batch 22 kirjutatud andmebaasi.
Batch 23 kirjutatud andmebaasi.
Batch 24 kirjutatud andmebaasi.
Batch 25 kirjutatud andmebaasi.
Batch 26 kirjutatud andmebaasi.
Batch 27 kirjutatud andmebaasi.
Batch 28 kirjutatud andmebaasi.
Batch 29 kirjutatud andmebaasi.
Batch 30 kirjutatud andmebaasi.
Batch 31 kirjutatud andmebaasi.
Batch 32 kirjutat

In [4]:
#query.stop()